# Run on Colab

1. Open this notebook in Colab: https://colab.research.google.com/github/JoshBong/latent-dispatch/blob/main/colab.ipynb (or upload the file). Pick an **L4** GPU under *Runtime → Change runtime type*, set the first code cell, then *Run all*.
2. The code is cloned from GitHub onto the Colab machine at `COMMIT`: `main`, or a commit hash to pin a run. The log prints the commit it runs, and every results row records it.
3. `LOAD` names finished runs in `archive/` to start from. They are checked and unpacked first, so only the questions they didn't answer are run, and the tarball at the end holds old and new answers together.
4. Results stay on that machine, not on Drive. When the run ends, or any time after one notebook has finished, download `/content/results-<model>-<dtype>.tar.gz` from the Files panel. Then disconnect and delete the runtime: an idle runtime keeps using compute units. To share a finished run, add its tarball to `archive/` under a dated name.

**Keep the tab open.** The Colab machine's disk is wiped when its runtime ends, and Colab Pro doesn't keep running after the browser disconnects (background execution is a Pro+ feature). Keep the laptop awake. Every question is saved as soon as it finishes, so *Run all* again on the same runtime picks up where it stopped, with the code that runtime already has.

Stages: machine check → code → install → earlier results → benchmarks → tests (including this model's real-weights check) → smoke run of every notebook + audit → full run → pack. Everything up to the tests stops the notebook on failure. The smoke gate leaves out any arm whose output is broken and any notebook that fails in its smoke; the run stops there only if b1 is broken. In the full run a failing notebook doesn't stop the next one, and results are packed after every notebook.

In [ ]:
# What to run. One model per run: check its results before moving to a bigger model.
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SEEDS = [0]
N_PER_CELL = 100                                  # quick preliminary; None = each notebook's full design
NOTEBOOKS = ["part0_ruler", "h1_encoding", "h2_thinking"]
SKIP = {"h2_thinking": ["t1_1024"]}               # arm names to leave out, per notebook
COMMIT = "main"                                   # the code to run: a branch, or a commit hash to pin the run
LOAD = []                                         # finished runs to start from, by name in archive/, e.g.
                                                  # ["Qwen3-4B-bfloat16-2026-11-13"]: their questions aren't rerun
RUN_TESTS = True
GATE_BOXED, GATE_CUT = 50.0, 20.0                 # smoke gate, for broken output only: an arm is left out if fewer
                                                  # than half its smoke answers are boxed or more than a fifth hit
                                                  # the token cap

ROOT = "/content/latent-dispatch"

In [ ]:
import os
import subprocess
import sys

import torch


def sh(cmd):
    """Run a shell command, show the end of its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.stdout.strip():
        print(r.stdout[-(6000 if r.returncode else 1500):].strip(), flush=True)
    if r.returncode:
        raise RuntimeError(f"failed: {cmd}")


print(sys.version)
assert torch.cuda.is_available(), "no GPU: connect to a Colab GPU server (L4)"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")
if DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise SystemExit("this GPU has no bf16 (a T4?): pick an L4 or A100")

In [ ]:
# The code: cloned from GitHub at COMMIT, once per machine. A machine that already has it keeps the commit it is on,
# so a rerun adds to the same results with the same code. To switch commits, delete /content/latent-dispatch first.
REPO = "https://github.com/JoshBong/latent-dispatch.git"
if os.path.exists(ROOT):
    print(f"{ROOT} is already on this machine: COMMIT is not applied again")
else:
    sh(f"rm -rf {ROOT}.tmp && git clone -q {REPO} {ROOT}.tmp && git -C {ROOT}.tmp checkout -q {COMMIT} "
       f"&& mv {ROOT}.tmp {ROOT}")
os.chdir(ROOT)
sh("git log -1 --format='running commit %h (%s)'")

In [ ]:
sh("pip install -q -e '.[dev]'")       # pins transformers 4.57.6; torch comes with Colab
sys.path.insert(0, ROOT)

In [ ]:
# Start from finished runs: each is checked, then unpacked into results/. Questions they answered aren't run again.
from latent_dispatch import results

for name in LOAD:
    results.load_archive(f"archive/{name}.tar.gz")

In [ ]:
names = ["toy", "toy_d0", "musique", "hotpotqa"]
missing = [n for n in names if not os.path.exists(f"benchmark/{n}.jsonl")]
if missing:
    sh("python -m latent_dispatch.data " + " ".join(missing))
sh("cd benchmark && sha256sum -c SHA256SUMS")

In [ ]:
if RUN_TESTS:
    sh("python -m pytest -q")                                                   # fast suite, ~1 min
    sh(f"MODEL={MODEL} DTYPE={DTYPE} DEVICE=cuda python -m pytest -q -m slow tests/test_real_model.py")

In [ ]:
# Smoke: every notebook on 2 questions per cell, then the audit of those answers. The gate only catches broken output
# (a wrong prompt or stop token gives ~0% boxed, ~100% cut): such an arm is left out of the full run, and the run
# stops only if b1, the reference every arm is compared to, is broken. Milder problems are printed below and show in
# the final audit. Only the smoke questions are judged, so loaded results or a rerun don't change the verdict.
import glob
import logging

import papermill as pm

from latent_dispatch import analysis, results

log = logging.getLogger("papermill")
log.setLevel(logging.INFO)
log.propagate = False
if not log.handlers:
    log.addHandler(logging.StreamHandler(sys.stdout))
os.makedirs("notebooks/runs", exist_ok=True)
tag = MODEL.split("/")[-1]
gate_done, dropped, failed = False, set(), {}


def why(e):
    """The error a notebook stopped on (papermill wraps it in a long message)."""
    return f"{e.ename}: {e.evalue}" if hasattr(e, "ename") else f"{type(e).__name__}: {str(e)[-500:]}"


def run_notebook(name, smoke):
    out = f"notebooks/runs/{tag}-{name}{'-smoke' if smoke else ''}.ipynb"
    pm.execute_notebook(f"notebooks/{name}.ipynb", out, kernel_name="python3", log_output=True,
                        parameters=dict(MODEL=MODEL, DTYPE=DTYPE, SMOKE=smoke, SEEDS=SEEDS, N_PER_CELL=N_PER_CELL,
                                        SKIP=SKIP.get(name, []) + sorted(dropped)))


for name in NOTEBOOKS:
    try:
        run_notebook(name, smoke=True)
    except Exception as e:                 # the failing cell is in notebooks/runs/<model>-<name>-smoke.ipynb
        failed[name] = why(e)
        print(f"\n!!! {name} failed in its smoke and is left out of the full run: {failed[name]}", flush=True)
saved = results.load_all(MODEL, DTYPE)
df = saved[saved.smoke.astype(bool) & (saved.seed == SEEDS[0])]
display(analysis.audit(df))

verdict = analysis.gate(df, GATE_BOXED, GATE_CUT)
unboxed, capped = analysis.output_flags(df)
for arm, v in verdict[(verdict.unboxed > 0) | (verdict.capped > 0)].iterrows():
    kind = "BROKEN, left out of the full run" if v.broken else "not judged, unanswerable by design" if v.exempt \
        else "warning"
    print(f"\n{kind}: {arm}: {v.unboxed} unboxed, {v.capped} hit the cap, of {v.n}")
    for _, r in df[(df.arm == arm) & (unboxed | capped)].head(3).iterrows():
        text = r.text if len(r.text) <= 450 else r.text[:300] + " [...] " + r.text[-150:]
        print(f"  --- {r.id}  gold={r.answers}\n  {text!r}")
dropped = set(verdict.index[verdict.broken])
for arm in sorted(dropped):                # nothing but its smoke answers saved: move them out of the tables' way
    if (saved.arm == arm).sum() == (df.arm == arm).sum():
        path = results.results_path(MODEL, DTYPE, arm)
        os.replace(path, f"{path}.left-out")
if "b1" in dropped:
    raise SystemExit("b1 is broken, so nothing can be compared: read the outputs above before running anything")
gate_done = True

In [ ]:
# Full run, one notebook after another. Resumes: questions already saved (the smoke's, and any loaded) are skipped. A
# notebook that fails doesn't stop the next one, and this model's results are packed after every notebook.
if not globals().get("gate_done"):
    raise SystemExit("run the smoke cell above first: it decides which arms are left out")


def pack():
    files = [f"results/{tag}-{DTYPE}"] + sorted(glob.glob(f"notebooks/runs/{tag}-*.ipynb"))
    sh(f"tar czf /content/results-{tag}-{DTYPE}.tar.gz " + " ".join(files))


for name in NOTEBOOKS:
    if name in failed:
        continue
    try:
        run_notebook(name, smoke=False)
    except Exception as e:                 # the failing cell is in notebooks/runs/<model>-<name>.ipynb
        failed[name] = why(e)
        print(f"\n!!! {name} failed, going on with the next notebook: {failed[name]}", flush=True)
    try:
        pack()
    except Exception as e:
        print(f"\n!!! packing the results failed: {e}", flush=True)
display(analysis.audit(results.load_all(MODEL, DTYPE)))

In [ ]:
print(f"download /content/results-{tag}-{DTYPE}.tar.gz from the Files panel, then disconnect and delete the runtime")
if dropped or failed:
    raise SystemExit(f"left out as broken: {sorted(dropped)}; notebooks that failed: {failed}")